In [0]:
!pip install feature_engine

In [0]:
# Random-Rorest
# Regressao-logistica
# Grandient-Boosting (XGBoost, LGBoost)
# MLP (multi layer perceptron)
# SVM
# Ada-Boost
# Cat-Boost

from sklearn import tree

In [0]:
# SAMPLE

from sklearn import model_selection

df_abt = spark.table("workspace.analytics.abt_ativacao").toPandas()
df_oot = df_abt[df_abt['dtRef']==df_abt['dtRef'].max()]

df_train_test = df_abt[df_abt['dtRef']!=df_abt['dtRef'].max()]

target = 'flAtivacao'
features = df_train_test.columns.tolist()[2:-1]
X = df_train_test[features]
y = df_train_test[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa respoata: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")


In [0]:
X_train.head()

In [0]:
# EXPLORE

is_na = X_train.isna().mean()
is_na[is_na>0]

# DiasUltimoStreak -> 28 dias

# qtdCursosIniciados -> 0
# qtdCursosFinalizados -> 0
# python_2025 -> 0
# plataforma_ml_2026 -> 0
# mlflow_2025 -> 0
# carreira -> 0
# nekt_2025 -> 0
# estatistica_2025 -> 0
# coleta_dados_2024 -> 0
# python_2024 -> 0
# lago_mago_2024 -> 0
# github_2025 -> 0
# trampar_lakehouse_2024 -> 0
# ds_databricks_2024 -> 0
# sql_2020 -> 0
# ds_pontos_2024 -> 0
# streamlit_2025 -> 0
# estatistica_2024 -> 0
# ml_2024 -> 0
# ragia -> 0
# sql_2025 -> 0
# loyalty_predict_2025 -> 0
# go_2026 -> 0
# f1_lake -> 0
# ia_canal_2025 -> 0
# tse_analytics_2024 -> 0
# machine_learning_2025 -> 0
# matchmaking_trampar_de_casa_2024 -> 0
# pandas_2024 -> 0
# pandas_2025 -> 0
# speed_f1 -> 0
# github_2024 -> 0

# avgTempoInicioUltimo -> max base
# avgTempoInicioFim -> max base

In [0]:
# MODIFY

from feature_engine import imputation

features_0 = [
    'qtdCursosIniciados',
    'qtdCursosFinalizados',
    'python_2025',
    'plataforma_ml_2026',
    'mlflow_2025',
    'carreira',
    'nekt_2025',
    'estatistica_2025',
    'coleta_dados_2024',
    'python_2024',
    'lago_mago_2024',
    'github_2025',
    'trampar_lakehouse_2024',
    'ds_databricks_2024',
    'sql_2020',
    'ds_pontos_2024',
    'streamlit_2025',
    'estatistica_2024',
    'ml_2024',
    'ragia',
    'sql_2025',
    'loyalty_predict_2025',
    'go_2026',
    'f1_lake',
    'ia_canal_2025',
    'tse_analytics_2024',
    'machine_learning_2025',
    'matchmaking_trampar_de_casa_2024',
    'pandas_2024',
    'pandas_2025',
    'speed_f1',
    'github_2024',
]

imput_0 = imputation.ArbitraryNumberImputer(variables=features_0, arbitrary_number=0)

imput_max_tail = imputation.EndTailImputer(variables=['DiasUltimoStreak','avgTempoInicioUltimo','avgTempoInicioFim'],
                                           imputation_method='max', fold=1)

# # AJUSTE DOS MÉTODOS
# imput_0.fit(X_train)
# imput_max_tail.fit(X_train)

# # APLICAÇÃO DOS MÉTODOS
# X_train_transform = imput_0.transform(X_train)
# X_train_transform = imput_max_tail.transform(X_train_transform)

In [0]:
# EXPLORE

# df_train = X_train_transform.copy()
# df_train[target] = y_train
# sumario = df_train.groupby(target)[features].mean().T
# sumario['rate'] = sumario[1] / sumario[0]
# sumario

In [0]:
# MODEL
import os
import logging
os.environ["MLFLOW_DATABRICKS_TAGS_TO_SKIP"] = "extraContext"
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

import mlflow
from sklearn import ensemble
from sklearn import linear_model
from sklearn import pipeline
from sklearn import metrics

# VINCULA O SCRIPT PYTHON COM O EXPERIMENTO DO MLFLOW
mlflow.set_experiment(experiment_id=1805667430415716)

with mlflow.start_run():

    mlflow.sklearn.autolog()

    # clf = linear_model.LogisticRegression()
    # clf = ensemble.RandomForestClassifier(min_samples_leaf=12, n_estimators=500, random_state=42)
    clf = ensemble.AdaBoostClassifier(n_estimators=1000, learning_rate=0.9)

    model_pipeline = pipeline.Pipeline(steps=[
        ('imputacao_0', imput_0),
        ('imput_max', imput_max_tail),
        ('classificador', clf) ])
    
    model_pipeline.fit(X_train, y_train)

    y_train_pred = model_pipeline.predict(X_train)
    y_train_prob = model_pipeline.predict_proba(X_train)[:,1]

    acc_train = metrics.accuracy_score(y_train, y_train_pred)
    auc_train = metrics.roc_auc_score(y_train, y_train_prob)

    y_test_pred = model_pipeline.predict(X_test)
    y_test_prob = model_pipeline.predict_proba(X_test)[:,1]

    # y_test_pred = (y_test_prob >= y_train.mean()).astype(int)

    acc_test = metrics.accuracy_score(y_test, y_test_pred)
    auc_test = metrics.roc_auc_score(y_test, y_test_prob)

    acc_teo_cu = metrics.accuracy_score(y_test, [0 for i in range(y_test.shape[0])])

    print(f"Acurácia Test: {acc_test:.4f} | Ganho: {acc_test / acc_teo_cu*100 - 100 :.2f}%")
    print(f"AUC Test: {auc_test:.4f} | Ganho: {auc_test/0.5*100 - 100 :.2f}%")

    mlflow.log_metrics( {"auc_train": auc_train, "auc_test": auc_test} )

In [0]:
import pandas as pd
pd.Series(clf.feature_importances_, index=X_train.columns.tolist()).sort_values(ascending=False)